In [ ]:
# Cell 1: Load GitHub PAT from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
pat = secrets.get_secret('GITHUB_PAT')
os.environ['GITHUB_PAT'] = pat
print('PAT loaded OK')

In [ ]:
# Cell 2: Clone repo (branch dengan fixes)
import os
pat = os.environ['GITHUB_PAT']

# cd to safe dir first to avoid getcwd error when rm -rf deletes cwd
%cd /kaggle/working
!rm -rf EMA-SKD
!git clone https://{pat}@github.com/almaas-izdihar/ema-skd EMA-SKD
%cd EMA-SKD
!git checkout experiment/ablation-baseline
!git log --oneline -5

In [ ]:
# Cell 3: Verify GPU
!nvidia-smi

In [ ]:
# Cell 4: Run 1 — Baseline (L_CE only, no EHSKD)
# Target paper: 75.55 ± 0.09
!python main.py \
  --data_type cifar100 \
  --data_path /kaggle/working/data \
  --classifier_type ResNet18 \
  --batch_size 128 \
  --end_epoch 2 \
  --workers 4 \
  --seed 2024 \
  --experiment_type run1_baseline

In [ ]:
# Cell 5: Run 2 — Full EMA-SKD (Fix A+B: α added to L_KD2 and L_Refine)
# Fix A: mixup_loss * args.weight (was × 1.0, now × 4.0)
# Fix B: weight2 default 1.0 → 4.0 (L_Refine now × 4.0)
# Target paper: 79.19 ± 0.15
!python main.py \
  --data_type cifar100 \
  --data_path /kaggle/working/data \
  --classifier_type ResNet18 \
  --batch_size 128 \
  --end_epoch 2 \
  --workers 4 \
  --seed 2024 \
  --beta 0.5 \
  --EHSKD \
  --experiment_type run2_emaskd_fixed_alpha

In [ ]:
# Cell 6: Evaluation & Visualization
import glob, re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

def parse_log(path):
    rows = []
    with open(path) as f:
        for line in f:
            if '[val]' not in line:
                continue
            def g(key):
                m = re.search(rf'\[{key} ([^\]]+)\]', line)
                return float(m.group(1)) if m else None
            rows.append({
                'epoch':      int(re.search(r'\[Epoch (\d+)\]', line).group(1)),
                'val_loss':   g('val_loss'),
                'top1':       g('val_top1_acc'),
                'top5':       g('val_top5_acc'),
                'ece':        g('ECE'),
                'aurc':       g('AURC'),
                'eaurc':      g('EAURC'),
            })
    return pd.DataFrame(rows).set_index('epoch')

def find_log(pattern):
    matches = sorted(glob.glob(f'models/{pattern}/log/log.txt'))
    if not matches:
        raise FileNotFoundError(f'No log found for pattern: {pattern}')
    return matches[-1]

baseline_log = find_log('*run1_baseline*')
fullskd_log  = find_log('*run2_emaskd_fixed_alpha*')

df_base = parse_log(baseline_log)
df_full = parse_log(fullskd_log)

print('Baseline log :', baseline_log)
print('EMA-SKD fixed:', fullskd_log)
print()

last_base = df_base.iloc[-1]
last_full = df_full.iloc[-1]

summary = pd.DataFrame({
    'Metric':    ['Top-1 Acc (%)', 'Top-5 Acc (%)', 'ECE (↓)', 'AURC×10³ (↓)', 'EAURC×10³ (↓)'],
    'Baseline':  [last_base.top1, last_base.top5, last_base.ece, last_base.aurc, last_base.eaurc],
    'EMA-SKD':   [last_full.top1, last_full.top5, last_full.ece, last_full.aurc, last_full.eaurc],
})
summary['Δ'] = summary['EMA-SKD'] - summary['Baseline']
print(summary.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print()
print(f'Paper targets  — Baseline: 75.55 ± 0.09  |  EMA-SKD: 79.19 ± 0.15')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('EMA-SKD (Fix A+B) vs Baseline — CIFAR-100 / ResNet18', fontsize=13)

for ax, (col, title) in zip(axes, [('top1','Top-1 Accuracy (%)'), ('val_loss','Val Loss'), ('ece','ECE (↓)')]):
    ax.plot(df_base.index, df_base[col], label='Baseline', marker='o', linewidth=1.5)
    ax.plot(df_full.index, df_full[col], label='EMA-SKD (fixed)', marker='s', linewidth=1.5)
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('eval_curves_fixed.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: eval_curves_fixed.png')

In [ ]:
# Cell 7: GPU Usage Analysis (from nvidia-smi --query-gpu log)
import glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

def parse_gpu_stats(path):
    # Format: "45 %, 2048 MiB, 16160 MiB"  (--query-gpu=utilization.gpu,memory.used,memory.total)
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = [p.strip().split()[0] for p in line.split(',')]
            if len(parts) < 3:
                continue
            try:
                rows.append({
                    'sm_pct':   int(parts[0]),
                    'fb_mb':    int(parts[1]),
                    'fb_total': int(parts[2]),
                })
            except ValueError:
                continue
    return pd.DataFrame(rows)

# find gpu_stats.log — pick latest run
gpu_logs = sorted(glob.glob('models/*/log/gpu_stats.log'))
if not gpu_logs:
    print('No gpu_stats.log found. Run training first.')
else:
    gpu_log = gpu_logs[-1]
    print(f'GPU log: {gpu_log}')
    df_gpu = parse_gpu_stats(gpu_log)

    if df_gpu.empty:
        print('gpu_stats.log found but no parseable rows.')
    else:
        fb_total = int(df_gpu.fb_total.iloc[0]) if 'fb_total' in df_gpu else 16160
        time_min = [i * 0.5 for i in range(len(df_gpu))]  # each sample = 30s = 0.5 min

        print(f'\nSamples: {len(df_gpu)} @ every 30s = ~{len(df_gpu)*30/60:.0f} min of monitoring')
        print(f'SM util   — mean: {df_gpu.sm_pct.mean():.1f}%  min: {df_gpu.sm_pct.min()}%  max: {df_gpu.sm_pct.max()}%')
        print(f'VRAM used — mean: {df_gpu.fb_mb.mean():.0f}MB  max: {df_gpu.fb_mb.max()}MB  / {fb_total}MB total')

        fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
        fig.suptitle('GPU Usage During Training (sampled every 30s)', fontsize=13)

        axes[0].plot(time_min, df_gpu.sm_pct, color='tab:orange', linewidth=1.2)
        axes[0].axhline(df_gpu.sm_pct.mean(), color='tab:orange', linestyle='--', alpha=0.6,
                        label=f'mean {df_gpu.sm_pct.mean():.1f}%')
        axes[0].set_ylabel('SM Utilization (%)')
        axes[0].set_ylim(0, 105)
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        axes[1].plot(time_min, df_gpu.fb_mb, color='tab:blue', linewidth=1.2)
        axes[1].axhline(fb_total, color='red', linestyle='--', alpha=0.4, label=f'Total ({fb_total}MB)')
        axes[1].set_ylabel('VRAM Used (MB)')
        axes[1].set_xlabel('Time (minutes)')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig('gpu_usage.png', dpi=120, bbox_inches='tight')
        plt.show()
        print('Saved: gpu_usage.png')

        print()
        mean_sm = df_gpu.sm_pct.mean()
        max_vram = df_gpu.fb_mb.max()
        if mean_sm < 50:
            print(f'⚠ GPU util rendah ({mean_sm:.1f}%) — kemungkinan bottleneck di dataloader. Coba naikkan --workers.')
        elif mean_sm > 85:
            print(f'✓ GPU util tinggi ({mean_sm:.1f}%) — GPU terpakai optimal.')
        else:
            print(f'~ GPU util moderate ({mean_sm:.1f}%) — acceptable.')
        if max_vram < 6000:
            print(f'✓ VRAM hanya {max_vram}MB dari {fb_total}MB — bisa naikkan --batch_size (256 atau 512).')
        elif max_vram > int(fb_total * 0.87):
            print(f'⚠ VRAM hampir penuh ({max_vram}MB) — jangan naikkan batch_size.')